# ISLES 2022 UNet


## Change Device to GPU

In [1]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print(torch.cuda.get_device_properties(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

Using device: cuda

Tesla T4
_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=15109MB, multi_processor_count=40)
Memory Usage:
Allocated: 0.0 GB
Cached:    0.0 GB


## Auxillary Functions

In [2]:
# Auxillary Functions

def downscale_tensor(tensor, size):
    # size = size you want tensor to be resized too
    
    from torchvision import transforms as transforms
    new_tnsr = torch.empty((2,1,size,size,size))
    # creates a function that resizes a tensor in the HxW (2D) domain 
    transform = transforms.Resize((size,size))
    
    for i, t in enumerate(tnsr):
        for n in t:        
            n = transform(n)
            n = torch.transpose(n, 0, 1)
            n = transform(n)
            n = torch.transpose(n, 0, 1)
            new_tnsr[i][0] = n
    return new_tnsr


def pad_tensor(tensor, size):
    # make a blank tensor of the correct size
    black_tensor = torch.zeros(2, 1, size, size, size)
    
    # find the current tensors size for HxWxD
    current_size = tensor[0][0].size()
    x,y,z = current_size
    
    offset_x = int((size - x) / 2)
    offset_y = int((size - y) / 2)
    offset_z = int((size - z) / 2)
    
    print(str(offset_x)+ ' ' +str(x+offset_x))
    print(str(offset_y)+ ' ' +str(y+offset_y))
    print(str(offset_z)+ ' ' +str(z+offset_z))
    
    black_tensor[:, :, offset_x:x+offset_x, offset_y:y+offset_y, offset_z:z+offset_z] = tensor 
    
    return black_tensor

def visualise_slice(tensor):
    for i, t in enumerate(new_tnsr):
        for n in t:
            plt.imshow(n[50], cmap='gray')
            plt.show()
            break
        break   

## Data Loading

In [3]:
import bidsio
bids_loader = bidsio.BIDSLoader(data_entities=[{'subject': '',
                                               'session': '',
                                               'suffix': 'T1w',
                                               'space': 'MNI152NLin2009aSym'}],
                                target_entities=[{'suffix': 'mask',
                                                'label': 'L',
                                                'desc': 'T1lesion'}],
                                data_derivatives_names=['ATLAS'],
                                target_derivatives_names=['ATLAS'],
                                batch_size=2,
                                root_dir='data/train/')

In [4]:
for data, target in bids_loader.load_batches():
    print(f'Our data has the shape {data.shape}')
    print(f'Our target has the shape {target.shape}')
    break

Our data has the shape (2, 1, 197, 233, 189)
Our target has the shape (2, 1, 197, 233, 189)


### shape = (batch, channel, X, Y, Z) 
x = sagittal plane
y = coronal plane
z = transverse plane

In [5]:
tensor = torch.tensor(data)
tensor.shape

torch.Size([2, 1, 197, 233, 189])

### Monai UNet

In [6]:
from monai.networks.nets import UNet

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=0,
).to(device)

### DIY UNet

In [7]:
from unet import UNet
model = UNet()

In [8]:
output = model(tensor)

after conv1
after conv2
after conv3
after conv4
after output


In [ ]:
output

## Predictions
Once your model is trained, you'll want to make predictions on the test data and upload them for evaluation. We expect the data to be formatted as a BIDS dataset. In this section, we'll show you how to easily format your predictions without having to go through the BIDS standard.  
First, we'll load the test data:

In [ ]:
import bidsio
bids_loader = bidsio.BIDSLoader(data_entities=[{'subject': '',
                                               'session': '',
                                               'suffix': 'T1w',
                                               'space': 'MNI152NLin2009aSym'}],
                                target_entities=[],
                                data_derivatives_names=['ATLAS'],
                                batch_size=4,
                                root_dir='data/test/')

In [ ]:
for dat, image_list in bids_loader.load_batch_for_prediction():
    print(f'Data shape: {dat.shape}')
    print(f'Example BIDS file: {image_list[0]}')
    break

You'll notice that we use a different generator for loading the predictions. This generator also yields the BIDS image file that stored the data. We'll create a new BIDS directory using this information.  
First, we'll need to create a model:

In [ ]:
# Create great model.
import numpy as np
class some_model():
    def __init__(self):
        '''
        Simple model to serve as an example.
        '''
        return
    
    def predict(self, data: np.ndarray) -> np.ndarray:
        '''
        Returns '1' for voxels whose value are greater than the image mean.
        Parameters
        ----------
        data : np.ndarray
            Data for which to make a prediction of the labels.
        Returns
        -------
        np.ndarray
            Model prediction for the input data.
    '''
        data_mean = np.mean(data)
        return np.array(data > data_mean, dtype=np.float32)
your_model = some_model()

The `your_model` object will be used a stand-in for a fully-trained model.  
As before, we'll use the `load_batch_for_prediction` method to obtain our data. We can write out our predictions as we generate them using the `write_image_like` method:

In [ ]:
help(bids_loader.write_image_like)

In [ ]:
example_output_dir = 'prediction_bids/'  # Directory where to write out predictions
for dat, image_list in bids_loader.load_batch_for_prediction():
    prediction = your_model.predict(dat)  # Make a prediction
    # Reduce to set of 3D images
    for i in range(prediction.shape[0]):  # Iterate through each sample in the batch
        pred_out = prediction[i,0,...]
        image_ref = image_list[i][0]
        print(f"Writing image for subject {image_ref.entities['subject']}")
        
        bids_loader.write_image_like(data_to_write=pred_out,
                                     image_to_imitate=image_ref,
                                     new_bids_root=example_output_dir,
                                     new_entities={'label': 'L',
                                                   'suffix': 'mask'})
    break

We see that we create a file for each subject present in our batch. Let's verify that the files were created.

In [ ]:
import os
for p, _, fnames in os.walk(example_output_dir):  # Walk through dir structure
    if(len(fnames) > 0):
        for f in fnames:
            print(os.path.join(p, f))  # Print full path of files that are found

You should see one image for each sample in a batch, with `label-L` and `mask` inserted into the filename. BIDS requires one more file, `dataset_description.json`, which we can create with `write_dataset_description`:

In [ ]:
help(bidsio.BIDSLoader.write_dataset_description)

In [ ]:
bidsio.BIDSLoader.write_dataset_description(bids_root=example_output_dir,
                                            dataset_name='atlas2_prediction',
                                            author_names=['Hutton, A.'])

We can then take a look at the JSON file:

In [ ]:
import json, os
f = open(f'{example_output_dir}{os.sep}dataset_description.json')
dataset_description = json.load(f)
f.close()
print(dataset_description)

Our predictions are now a BIDS-compatible dataset and can be zipped and submitted to the Grand Challenge website.

In [ ]:
import bids
prediction_bids = bids.BIDSLayout(root=example_output_dir, derivatives=example_output_dir)
print(prediction_bids.derivatives['atlas2_prediction'])